In [1]:
import torch
import torch.nn as nn

class Lite3x3(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.Conv2d(out_channels, out_channels, 3, 
                      padding=1, groups=out_channels, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)


In [2]:

class AggregationGate(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        w = self.gate(x)
        w = w.unsqueeze(-1).unsqueeze(-1)
        return x * w

In [3]:
class OSNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.stream1 = Lite3x3(out_channels, out_channels)
        self.stream2 = nn.Sequential(
            Lite3x3(out_channels, out_channels),
            Lite3x3(out_channels, out_channels)
        )
        self.stream3 = nn.Sequential(
            Lite3x3(out_channels, out_channels),
            Lite3x3(out_channels, out_channels),
            Lite3x3(out_channels, out_channels)
        )
        self.stream4 = nn.Sequential(
            Lite3x3(out_channels, out_channels),
            Lite3x3(out_channels, out_channels),
            Lite3x3(out_channels, out_channels),
            Lite3x3(out_channels, out_channels)
        )
        self.ag = AggregationGate(out_channels)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.skip = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ) if in_channels != out_channels else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.skip(x)
        out = self.conv1(x)
        s1 = self.ag(self.stream1(out))
        s2 = self.ag(self.stream2(out))
        s3 = self.ag(self.stream3(out))
        s4 = self.ag(self.stream4(out))
        out = s1 + s2 + s3 + s4
        out = self.conv2(out)
        out = self.relu(out + identity)
        return out

In [4]:
class OSNet(nn.Module):
    def __init__(self, num_classes=751):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.pool = nn.MaxPool2d(3, stride=2, padding=1)
        self.conv2 = OSNetBlock(64, 256)
        self.conv3 = OSNetBlock(256, 384)
        self.conv4 = OSNetBlock(384, 512)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [5]:
model = OSNet(num_classes=751)
x = torch.randn(2, 3, 256, 128)
out = model(x)
print(f"OSNet 구현 완료!")
print(f"입력: {x.shape} → 출력: {out.shape}")
total_params = sum(p.numel() for p in model.parameters())
print(f"총 파라미터 수: {total_params/1e6:.1f}M")
print(f"논문 파라미터 수: 2.2M")

OSNet 구현 완료!
입력: torch.Size([2, 3, 256, 128]) → 출력: torch.Size([2, 512])
총 파라미터 수: 6.7M
논문 파라미터 수: 2.2M
